In [1]:
import os
os.chdir('/workspace/b0d2b5bc-026b-42e9-9ab3-84ab4728dd30')
print(os.listdir('.'))


['svm_norm_features_results.json', '.config', '.kernel_llm_logs_1.txt', ' v6 — Binding Context Document.pdf', 'memory', 'peaks_features_F1_F12.csv', '.prompts']


In [2]:
import json
import pandas as pd

with open('svm_norm_features_results.json') as f:
 norm_results = json.load(f)
print(json.dumps(norm_results, indent=2)[:3000])


{
 "protocol": "class-held-out: train on F1,F4,F6 -> test on remaining",
 "train_classes": [
 "F1",
 "F4",
 "F6"
 ],
 "test_classes": [
 "F2",
 "F5p",
 "F5m",
 "F7",
 "F9",
 "F10",
 "F11",
 "F12"
 ],
 "feature_set_original": [
 "feat_xi",
 "feat_A_log_peak",
 "feat_B_curvature",
 "feat_C_roughness"
 ],
 "feature_set_normalized": [
 "norm_xi",
 "norm_amplitude",
 "norm_curvature",
 "norm_roughness"
 ],
 "normalization_laws": {
 "norm_amplitude": "feat_A_log_peak / sqrt(0.5*d*log(d*log(t+3)+log(q))) (Selberg CLT scaling)",
 "norm_curvature": "feat_B_curvature * (2\u03c0/(d*log(t+3)+log(q)))^2 (curvature dedimensionalized by mean spacing^2)",
 "norm_roughness": "feat_C_roughness * (2\u03c0/(d*log(t+3)+log(q)))^2",
 "norm_xi": "feat_xi (already dimensionless)"
 },
 "class_invariants": {
 "F1": {
 "q": 1,
 "d": 1
 },
 "F2": {
 "q": 5,
 "d": 1
 },
 "F4": {
 "q": 5,
 "d": 1
 },
 "F5p": {
 "q": 5,
 "d": 1
 },
 "F5m": {
 "q": 5,
 "d": 1
 },
 "F6": {
 "q": 3,
 "d": 1
 },
 "F7": {
 "q": 7,
 "d": 

In [3]:
df = pd.read_csv('peaks_features_F1_F12.csv')
print(df.shape)
print(df.head())
print(df['class'].value_counts())
print(df.columns.tolist())


(2200, 7)
 t function GRH_label feat_xi feat_A_log_peak feat_B_curvature \
0 22299.04 F1 1 0.473426 3.044499 4.557116 
1 24329.62 F1 1 0.473426 3.064509 4.660456 
2 26296.93 F1 1 0.473426 3.030568 4.810800 
3 28327.24 F1 1 0.473426 3.037365 4.988046 
4 28807.75 F1 1 0.473426 3.020980 5.017412 

 feat_C_roughness 
0 23.696544 
1 24.100112 
2 23.807971 
3 24.491607 
4 24.061750 


KeyError: 'class'

In [4]:
print(df['function'].value_counts())
print("Total peaks:", len(df))
print("Unique classes:", df['function'].unique())


function
F1 200
F2 200
F4 200
F5p 200
F5m 200
F6 200
F7 200
F9 200
F10 200
F11 200
F12 200
Name: count, dtype: int64
Total peaks: 2200
Unique classes: ['F1' 'F2' 'F4' 'F5p' 'F5m' 'F6' 'F7' 'F9' 'F10' 'F11' 'F12']


In [5]:
# Load the peaks data and identify what we need
df = pd.read_csv('peaks_features_F1_F12.csv')
print("Class -> GRH label:")
print(df.groupby('function')['GRH_label'].first())
print()
print("Peak t range per class:")
print(df.groupby('function')['t'].agg(['min','max','count']))


Class -> GRH label:
function
F1 1
F10 1
F11 1
F12 0
F2 1
F4 0
F5m 0
F5p 0
F6 1
F7 1
F9 1
Name: GRH_label, dtype: int64

Peak t range per class:
 min max count
function 
F1 22299.04 99937.90 200
F10 26100.91 99933.04 200
F11 18116.11 99843.04 200
F12 16937.47 99997.39 200
F2 17859.88 99958.87 200
F4 17859.79 99972.82 200
F5m 17859.79 99972.82 200
F5p 17859.79 99972.82 200
F6 15387.40 99716.77 200
F7 14735.53 99923.68 200
F9 21499.30 99890.56 200


In [6]:

# Plan: For each of 11 classes, generate Dirichlet coefficients a_n up to N=1e6.
# Then for each peak t, compute S_k(t; N) = sum_{n: omega(n)=k} a_n / n^{1/2 + i t}.
# Build features [log|S_1|, ..., log|S_5|], normalize, run held-out SVM.

# First, generate omega(n) for n up to N=1e6 using a sieve.
import numpy as np
from numba import njit, prange

N_MAX = 10**6

@njit(cache=True)
def compute_omega(N):
 # smallest prime factor sieve, then count distinct primes per n
 spf = np.zeros(N+1, dtype=np.int64)
 for i in range(2, N+1):
 if spf[i] == 0:
 for j in range(i, N+1, i):
 if spf[j] == 0:
 spf[j] = i
 omega = np.zeros(N+1, dtype=np.int8)
 for n in range(2, N+1):
 m = n
 last_p = 0
 c = 0
 while m > 1:
 p = spf[m]
 if p != last_p:
 c += 1
 last_p = p
 m //= p
 omega[n] = c
 return omega, spf

omega, spf = compute_omega(N_MAX)
print("omega computed. Distribution:", np.bincount(omega[1:]))
print("Max omega:", omega.max())


omega computed. Distribution: [ 1 78734 288726 379720 208034 42492 2285 8]
Max omega: 7


In [7]:

# Now build coefficient arrays for each class. We will use degree-1 functions where we can
# compute a_n directly (multiplicatively). For F9, F10, F11 (modular L-functions), we'll need 
# to attempt LMFDB retrieval. Let's first define the simpler classes:
# 
# F1: zeta -> a_n = 1
# F2: chi_4 mod 5 (Note: actually chi mod 5 with order 4, the unique character of order 4)
# F4: L_DH = 1/sqrt(2+2*kappa^2) * sum chi5_components -- specifically the Davenport-Heilbronn 
# L_DH(s) = (1-i*tan(theta))/2 * L(s,chi5_a) + (1+i*tan(theta))/2 * L(s,chi5_b),
# where chi5_a, chi5_b are characters mod 5 of order 4.
# F5p, F5m: L_DH(epsilon=+/-0.05)
# F6: Liouville lambda(n) = (-1)^Omega(n)
# F7: Mobius mu(n)
# F12: a constructed RH-violator
# 
# For F9 (Ramanujan Delta), F10 (modular form level 11/17/23, weight 2/4), F11 (Sym^2/Sym^3 Delta):
# We'd need LMFDB. Per the dataset description, this is a known issue.
#
# Given complexity of building all classes correctly from scratch, the key issue: the existing 
# peaks_features_F1_F12.csv was generated based on these classes. The peak locations are 
# already there. We need to compute S_k at those locations.
#
# Let me build the coefficients carefully. Start with what we can build cleanly.

import numpy as np
from sympy import primerange

# Liouville lambda
@njit(cache=True)
def compute_liouville(N, spf):
 lam = np.ones(N+1, dtype=np.int8)
 lam[0] = 0
 for n in range(2, N+1):
 m = n
 Omega = 0
 while m > 1:
 p = spf[m]
 while m % p == 0:
 m //= p
 Omega += 1
 lam[n] = -1 if (Omega % 2) else 1
 return lam

# Mobius
@njit(cache=True)
def compute_mobius(N, spf):
 mu = np.ones(N+1, dtype=np.int8)
 mu[0] = 0
 for n in range(2, N+1):
 m = n
 Omega = 0
 squarefree = True
 while m > 1:
 p = spf[m]
 cnt = 0
 while m % p == 0:
 m //= p
 cnt += 1
 if cnt > 1:
 squarefree = False
 break
 Omega += 1
 if not squarefree:
 mu[n] = 0
 else:
 mu[n] = -1 if (Omega % 2) else 1
 return mu

lam = compute_liouville(N_MAX, spf)
mu = compute_mobius(N_MAX, spf)
print("Liouville sample:", lam[:20])
print("Mobius sample:", mu[:20])


Liouville sample: [ 0 1 -1 -1 1 -1 1 -1 -1 1 1 -1 -1 -1 1 1 1 -1 -1 -1]
Mobius sample: [ 0 1 -1 -1 0 -1 1 -1 0 0 1 -1 0 -1 1 1 0 -1 0 -1]


In [8]:

# Dirichlet character mod 5 of order 4: chi(2) = i (primitive)
# chi5(n): 1 if n=1 mod 5, i if n=2 mod 5, -i if n=3 mod 5, -1 if n=4 mod 5, 0 if n=0 mod 5
# Define mod 4 (chi_4 quadratic character mod 4) — but per F2 it says "L(s,chi_4), chi_4 mod 5".
# That's confusing notation -- it's likely the order-4 character mod 5.

def chi5_a(n_arr):
 # Generates chi(n) for n in array, char mod 5 of order 4 with chi(2)=i
 # Multiplicatively: chi(2)=i, chi(3)=-i, chi(4)=-1, chi(1)=1
 # n mod 5: 0 -> 0, 1 -> 1, 2 -> i, 3 -> -i, 4 -> -1
 out = np.zeros(len(n_arr), dtype=np.complex128)
 r = n_arr % 5
 out[r==1] = 1.0
 out[r==2] = 1j
 out[r==3] = -1j
 out[r==4] = -1.0
 return out

def chi5_b(n_arr):
 # Conjugate character: chi(2) = -i
 out = np.zeros(len(n_arr), dtype=np.complex128)
 r = n_arr % 5
 out[r==1] = 1.0
 out[r==2] = -1j
 out[r==3] = 1j
 out[r==4] = -1.0
 return out

# real character chi_3 mod 3 (quadratic): chi(1)=1, chi(2)=-1, chi(0)=0
def chi3(n_arr):
 out = np.zeros(len(n_arr), dtype=np.float64)
 r = n_arr % 3
 out[r==1] = 1.0
 out[r==2] = -1.0
 return out

# real character mod 7 (quadratic): the Legendre symbol (n/7)
# Squares mod 7: 1, 2, 4 -> chi=1; non-squares: 3, 5, 6 -> chi=-1
def chi7(n_arr):
 out = np.zeros(len(n_arr), dtype=np.float64)
 r = n_arr % 7
 out[(r==1) | (r==2) | (r==4)] = 1.0
 out[(r==3) | (r==5) | (r==6)] = -1.0
 return out

# real character mod 5: Legendre symbol — quadratic char mod 5: chi(1)=1, chi(2)=-1, chi(3)=-1, chi(4)=1
def chi5_quadratic(n_arr):
 out = np.zeros(len(n_arr), dtype=np.float64)
 r = n_arr % 5
 out[(r==1) | (r==4)] = 1.0
 out[(r==2) | (r==3)] = -1.0
 return out

n_arr = np.arange(N_MAX+1)
chi5a = chi5_a(n_arr)
chi5b = chi5_b(n_arr)
chi5_quad = chi5_quadratic(n_arr)
chi3_arr = chi3(n_arr)
chi7_arr = chi7(n_arr)

# Davenport-Heilbronn function:
# L_DH(s) = (1-i*tan(theta))/2 * L(s,chi5_a) + (1+i*tan(theta))/2 * L(s,chi5_b)
# where tan(theta) = (sqrt(10-2*sqrt(5))-2)/(sqrt(5)-1) per Titchmarsh
# Equivalent: kappa = tan(theta), and validated value kappa ≈ 0.28408
import math
kappa = 0.28408 # validated DH constant

# a_n for L_DH = (1 - i*kappa)/2 * chi5_a(n) + (1+i*kappa)/2 * chi5_b(n)
a_F4 = (1 - 1j*kappa)/2 * chi5a + (1 + 1j*kappa)/2 * chi5b

# perturbations: F5p kappa+0.05, F5m kappa-0.05
kp = kappa + 0.05
km = kappa - 0.05
a_F5p = (1 - 1j*kp)/2 * chi5a + (1 + 1j*kp)/2 * chi5b
a_F5m = (1 - 1j*km)/2 * chi5a + (1 + 1j*km)/2 * chi5b

# Verify: at n=2, chi5_a=i, chi5_b=-i. a_n = (1-ik)/2 * i + (1+ik)/2 * (-i) = (i+k)/2 + (-i+k)/2 = k
# So a_2 = kappa. Good.
print("a_F4[2] = ", a_F4[2], "should be approx", kappa)
print("a_F4[3] = ", a_F4[3], "should be approx", -kappa)
print("a_F4[4] = ", a_F4[4], "should be -1")
print("a_F4[5] = ", a_F4[5], "should be 0")


a_F4[2] = (0.28408+0j) should be approx 0.28408
a_F4[3] = (-0.28408+0j) should be approx -0.28408
a_F4[4] = (-1+0j) should be -1
a_F4[5] = 0j should be 0


In [9]:

# F12: "L(s,χ_3)+L(s,χ_5) and 4 other DH-style linear combos" — constructed RH-violators.
# Simple choice: L(s,chi3) + L(s,chi5_quadratic). 
# This is multiplicative? No, sum of two L-functions is generally NOT multiplicative.
# But the coefficients are well-defined: a_n = chi3(n) + chi5_quadratic(n).
a_F12 = chi3_arr + chi5_quad

# F2: L(s, chi_4) where chi_4 is "mod 5" — interpretation: order-4 char mod 5 (the chi5_a above).
# Use chi5_a as F2.
a_F2 = chi5a.copy()

# F1: zeta. a_n = 1 for n>=1.
a_F1 = np.ones(N_MAX+1, dtype=np.complex128)
a_F1[0] = 0.0

# F6: Liouville
a_F6 = lam.astype(np.complex128)
a_F6[0] = 0.0

# F7: Mobius
a_F7 = mu.astype(np.complex128)
a_F7[0] = 0.0

# Now F9, F10, F11 -- modular L-functions. We need LMFDB coefficients.
# Per the data description:
# F9: Ramanujan Δ (level 1, weight 12) - coefficients = tau(n)/n^{11/2} (analytic normalization)
# F10: weight-2/4 newform on Γ_0(11/17/23) - similar normalization
# F11: Sym^2/Sym^3 of Δ - analytic normalization
#
# The (q,d) assignments tell us:
# F9: q=1, d=2 -- this is L(s, Delta) with analytic normalization. Conductor=1, degree=2.
# F10: q=11, d=2 -- this is the level-11 weight-2 newform (elliptic curve modular form)
# F11: q=1, d=3 -- this is Sym^2 of Delta, level 1 degree 3
# Coefficients need to be retrieved from LMFDB.

# These are challenging to retrieve and validate. Given the complexity, let's check if we can
# get them from LMFDB via API.
print("Need to retrieve modular form coefficients. Will attempt LMFDB.")


Need to retrieve modular form coefficients. Will attempt LMFDB.


In [10]:

# Try to retrieve Ramanujan tau coefficients up to N=1e6.
# The LMFDB has Delta = 1.1.a.a (level 1, weight 12). 
# Tau(n) is multiplicative. tau(p^k) = tau(p)*tau(p^{k-1}) - p^{11}*tau(p^{k-2})
# So we just need tau at primes up to N. There are ~78498 primes below 1e6.
# 
# Let's try LMFDB API directly for tau values at primes. Alternative: use sympy?
# Actually, sympy has nothing direct. Let's use a known list of tau values, or compute them
# from the q-expansion of Delta = q*prod(1-q^n)^24.

# Compute q-expansion of Delta up to N_MAX terms.
# Delta(q) = q * prod_{n>=1} (1-q^n)^24
# Coefficient of q^n in Delta = tau(n)
# 
# For N=1e6 this requires O(N^2) work naively. With Euler-style operations, 
# we can use the recursion Delta = q * eta(q)^24, where eta = prod(1-q^n).
# A standard way: compute eta(q)^24 by the convolution. But this is O(N^2)... too expensive for 1e6.
# 
# Alternative: only need tau(p) for p prime up to N. Then build tau(n) multiplicatively.
# But tau(p) computation requires summing... more efficient method needed.
# 
# Use Hecke: tau(p) = sigma_11(p) * something? No. 
# Actually, the standard efficient way: compute eta^24 via power series with truncation N.
# Convolution: c[n] = sum_{k=0..n} a[k]*b[n-k]. For 24-fold product done as sequential 
# convolutions, total work is 24 * N * (avg sparsity).
# 
# Eta = sum (-1)^k q^{k(3k-1)/2} (Euler's pentagonal). So eta has only ~sqrt(2N/3) terms 
# nonzero up to N. Eta^2, Eta^3, etc. -- each subsequent convolution could be done efficiently.

# Actually simpler: use known closed-form. Use sage or external. Let me try scrape LMFDB API.

import requests
# Try LMFDB API for tau coefficients
try:
 r = requests.get("https://www.lmfdb.org/api/mf_newforms/?label=1.12.a.a&_format=json", timeout=20)
 print(r.status_code, r.text[:500])
except Exception as e:
 print("Error:", e)


200 {
 "table": "mf_newforms",
 "timestamp": "2026-05-09T03:26:55.743785",
 "data": [
 {
 "id": 0,
 "Nk2": 144,
 "analytic_conductor": 0.7683431805595888,
 "analytic_rank": 0,
 "analytic_rank_proved": true,
 "atkin_lehner_eigenvals": [],
 "atkin_lehner_string": "",
 "char_conductor": 1,
 "char_degree": 1,
 "char_is_minimal": true,
 "char_is_real": true,
 "char_orbit_index": 1,
 "char_orbit_label": "a",
 "char_order": 1,
 


In [11]:

# Try mf_hecke_traces or mf_hecke_nf for actual coefficients
try:
 r = requests.get("https://www.lmfdb.org/api/mf_hecke_traces/?hecke_orbit_code=1.12.a.a&_format=json&n=1..30", timeout=20)
 print(r.status_code, r.text[:1500])
except Exception as e:
 print("Error:", e)


404 <!DOCTYPE html>
<html lang="en">
 <head>
 <meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
 <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
 <meta name="description" content="Welcome to the LMFDB, the database of L-functions, modular forms, and related objects. These pages are intended to be a modern handbook including tables, formulas, links, and references for L-functions and their underlying objects."/>
 <title>LMFDB - LMFDB Page Not Found </title>
 <link id="style_css" href="/style.css" rel="stylesheet" type="text/css" /> 

 <!-- jQuery -->
 <script type="text/javascript"
 src="https://ajax.googleapis.com/ajax/libs/jquery/3.2.1/jquery.min.js"
 integrity="sha384-xBuQ/xzmlsLoJpyjoggmTEz8OWUFM0/RC5BsqQBDX2v5cMvDHcMakNTNrHIW2I5f"
 crossorigin="anonymous"></script>
 <!-- jQuery UI -->
 <script type="text/javascript"
 src="https://ajax.googleapis.com/ajax/libs/jqueryui/1.12.1/jquery-ui.min.js"
 integrity="sha384-Dziy8F2VlJQLMShA6FHWNul/veM9bCkRUaLqr1

In [12]:

# Let me compute tau directly via the q-expansion convolution efficiently using numba.
# Eta(q) = sum_{k=-inf..inf} (-1)^k q^{k(3k-1)/2}
# We need convolution. Eta^24 then multiplied by q gives Delta.
# Approach: keep eta_powers as numpy arrays. Do 24-fold convolution.

# Actually the more efficient approach: Eta^24 = Delta/q. We can use the Newman formula:
# Or compute eta as sparse, then square repeatedly:
# eta^2, eta^4, eta^8, eta^16, eta^24 = eta^16 * eta^8.
# Each squaring is convolution of full vectors but uses FFT.
# With N=1e6, FFT would take ~N*log(N) = 2e7 ops per FFT, times maybe 24 multiplies. Doable.

# Use numpy.fft for convolution of polynomials. But since coefficients can grow large, use 
# float64 may lose precision. tau(n) grows like O(n^{11/2 + eps}), so tau(1e6) ~ (1e6)^5.5 = 1e33. 
# This exceeds float64 precision. We need exact integer arithmetic, or work with the analytic 
# normalization.
# 
# The analytic normalization is a_n = tau(n)/n^{11/2}. Using this, |a_n| <= d(n) by Deligne. 
# So in analytic normalization, coefficients are bounded by ~d(n) ~ log(n).
# 
# Approach: compute lambda(p) = tau(p)/p^{11/2} for primes p, then build a_n multiplicatively.
# This requires computing tau(p) exactly. We can do this for primes p up to N=1e6 using 
# the q-expansion.

# But tau(p) for p~1e6 has magnitude up to 2*p^{11/2} ~ 2e33, requiring big integers.

# Strategy: use python big ints for the q-expansion convolution, but limited to primes only.
# Compute Delta mod nothing, full integers. The convolution costs:
# (Eta squarefold approach) — too slow with python ints for N=1e6.

# Alternative: avoid the full 1e6. Use a smaller N. The research objective allows for the 
# largest possible subset documented as failure if retrieval fails for a class.
# 
# But for the existing peaks_features_F1_F12.csv, peaks were already detected for F9, F10, F11.
# That means some prior computation was done. Let me check memory/notes for whether there are 
# saved coefficients.


In [13]:

# Strategy decision:
# Computing modular L-function coefficients up to N=10^6 from scratch within the time budget 
# is feasible but expensive. Let me try to retrieve them via LMFDB API search.
# Try mf_newforms field 'traces' which often contains a vector of Hecke trace values.

# Get Delta info with all fields
try:
 r = requests.get("https://www.lmfdb.org/api/mf_newforms/?label=1.12.a.a&_format=json", timeout=30)
 data = r.json()
 fields = list(data['data'][0].keys())
 print("Fields:", fields)
except Exception as e:
 print("Error:", e)


Fields: ['id', 'Nk2', 'analytic_conductor', 'analytic_rank', 'analytic_rank_proved', 'atkin_lehner_eigenvals', 'atkin_lehner_string', 'char_conductor', 'char_degree', 'char_is_minimal', 'char_is_real', 'char_orbit_index', 'char_orbit_label', 'char_order', 'char_parity', 'char_values', 'cm_discs', 'conrey_index', 'dim', 'field_disc', 'field_disc_factorization', 'field_poly', 'field_poly_is_cyclotomic', 'field_poly_is_real_cyclotomic', 'field_poly_root_of_unity', 'fricke_eigenval', 'has_non_self_twist', 'hecke_cutters', 'hecke_orbit', 'hecke_orbit_code', 'hecke_ring_generator_nbound', 'hecke_ring_index', 'hecke_ring_index_factorization', 'hecke_ring_index_proved', 'inner_twist_count', 'inner_twists', 'is_cm', 'is_largest', 'is_maximal', 'is_polredabs', 'is_rm', 'is_self_dual', 'is_self_twist', 'is_twist_minimal', 'label', 'level', 'level_is_powerful', 'level_is_prime', 'level_is_prime_power', 'level_is_prime_square', 'level_is_square', 'level_is_squarefree', 'level_primes', 'level_radica

In [14]:

import requests
r = requests.get("https://www.lmfdb.org/api/mf_newforms/?label=1.12.a.a&_format=json", timeout=30)
data = r.json()
traces = data['data'][0]['traces']
print("Length of traces:", len(traces))
print("First 20:", traces[:20])
# Traces are tau(n) for n=1,2,...


Length of traces: 1000
First 20: [1, -24, 252, -1472, 4830, -6048, -16744, 84480, -113643, -115920, 534612, -370944, -577738, 401856, 1217160, 987136, -6905934, 2727432, 10661420, -7109760]


In [15]:

# LMFDB only returns 1000 traces. We need up to N=1e6 for tau (Delta). 
# tau(p) for p > 1000 isn't stored directly via this API.
# 
# Strategy: compute tau(n) ourselves via q-expansion.
# For Delta = q * eta(q)^24, with eta(q) = sum (-1)^k q^{k(3k-1)/2} via Euler's pentagonal.
# 
# Plan: compute eta truncated to degree N as sparse vector, 
# then square it: eta^2 = polynomial multiplication. Keep doing convolutions.
# Better: use FFT. But float64 cannot handle tau(1e6) ~ 1e33.
# 
# Alternative: since we only need tau(n) for n up to 1e6, and we want to use the analytic 
# normalization a_n = tau(n)/n^{11/2}, we can work directly with the analytic-normalized 
# coefficients via the following:
# 
# Define b_p = tau(p)/p^{11/2}, |b_p| <= 2 by Deligne. Then b_n is multiplicative with 
# b_{p^k} = b_p * b_{p^{k-1}} - b_{p^{k-2}} (analytic normalization Hecke relation).
# 
# To get b_p we need tau(p) exactly. 
# Strategy: compute tau(n) for prime n up to N using a sieve approach. For each prime p, 
# compute tau(p) from eta^24 expansion limited to degree p (only need [q^p] coefficient).
# 
# This is still expensive but more tractable. The trick: extract one coefficient at a time.
# tau(p) = [q^p] of eta(q)^24 / q-shift, i.e., coefficient of q^{p-1} in eta(q)^24.
# 
# We can get eta(q)^24 as full power series by FFT-based convolution if we accept precision loss,
# OR by exact integer convolution using NumPy with object dtype (slow), OR by recursion.
# 
# Recursion (Faber polynomials, no good).
# 
# Practical approach: compute eta^24 mod a few large primes (CRT) and reconstruct.
# 
# Given runtime budget, let me try the simpler approach: use the trace API to get tau values up 
# to some range, then check if the LMFDB has another endpoint with more data.
# 
# Try: mf_newforms has 'traces' — but only 1000. There's probably an mf_hecke_traces endpoint 
# with primes. Let me check.

# Check if there's a separate API for full hecke data:
r = requests.get("https://www.lmfdb.org/api/?_format=json", timeout=20)
print(r.text[:3000])


<!DOCTYPE html>
<html lang="en">
 <head>
 <meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
 <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
 <meta name="description" content="Welcome to the LMFDB, the database of L-functions, modular forms, and related objects. These pages are intended to be a modern handbook including tables, formulas, links, and references for L-functions and their underlying objects."/>
 <title>LMFDB - API </title>
 <link id="style_css" href="/style.css" rel="stylesheet" type="text/css" /> 

 <!-- jQuery -->
 <script type="text/javascript"
 src="https://ajax.googleapis.com/ajax/libs/jquery/3.2.1/jquery.min.js"
 integrity="sha384-xBuQ/xzmlsLoJpyjoggmTEz8OWUFM0/RC5BsqQBDX2v5cMvDHcMakNTNrHIW2I5f"
 crossorigin="anonymous"></script>
 <!-- jQuery UI -->
 <script type="text/javascript"
 src="https://ajax.googleapis.com/ajax/libs/jqueryui/1.12.1/jquery-ui.min.js"
 integrity="sha384-Dziy8F2VlJQLMShA6FHWNul/veM9bCkRUaLqr199K94ntO5QUrLJBEbYegd

In [16]:

# Check L-functions endpoint for Delta - it may have more terms
# /L/api endpoint?
try:
 r = requests.get("https://www.lmfdb.org/api/lfunc_search/?label=4.1.1.0.0.0&_format=json", timeout=20)
 print(r.status_code, r.text[:1000])
except Exception as e:
 print("Error:", e)


200 <!doctype html><html lang="en-US" dir="ltr"><head><base href="https://www.google.com/recaptcha/challengepage/"><link rel="preconnect" href="//www.gstatic.com"><meta name="referrer" content="origin"><script nonce="khPW_AVcAu5f-QPmsP5RQQ">window['ppConfig'] = {productName: 'RecaptchaChallengePageUi', deleteIsEnforced: true , sealIsEnforced: true , heartbeatRate: 0.5 , periodicReportingRateMillis: 60000.0 , disableAllReporting: false };(function(){'use strict';function k(a){var b=0;return function(){return b<a.length?{done:!1,value:a[b++]}:{done:!0}}}function l(a){var b=typeof Symbol!="undefined"&&Symbol.iterator&&a[Symbol.iterator];if(b)return b.call(a);if(typeof a.length=="number")return{next:k(a)};throw Error(String(a)+" is not an iterable or ArrayLike");}var m=typeof Object.defineProperties=="function"?Object.defineProperty:function(a,b,c){if(a==Array.prototype||a==Object.prototype)return a;a[b]=c.value;return a};
function n(a){a=["object"==typeof globalThis&&globalThis,a,"object


In [17]:

# We're getting blocked. Strategy decision:
# Computing tau(n) for n up to 1e6 via q-expansion with exact arithmetic in Python is too slow.
# Instead, we'll use a clever workaround: compute tau(n) using sympy's etaproduct or similar.
#
# Actually, sympy doesn't have a fast tau implementation. The pari/gp library does (via cypari2)
# but we'd need it installed.

import importlib.util
for mod in ['cypari2', 'cypari', 'sage', 'sympy', 'flint', 'mpmath']:
 spec = importlib.util.find_spec(mod)
 print(mod, ':', 'found' if spec else 'not found')


cypari2 : not found
cypari : not found
sage : not found
sympy : found
flint : not found
mpmath : found


In [18]:

# Let's try installing cypari2 (PARI/GP Python interface) -- it has fast tau
import subprocess
res = subprocess.run(['pip', 'install', '-q', 'cypari2'], capture_output=True, text=True, timeout=180)
print("stdout:", res.stdout[-500:])
print("stderr:", res.stderr[-500:])


stdout: 
stderr: 


In [19]:

import cypari2
pari = cypari2.Pari()
# tau(n) using PARI
print("tau(1)=", pari.ramanujantau(1))
print("tau(2)=", pari.ramanujantau(2))
print("tau(3)=", pari.ramanujantau(3))
print("tau(1000)=", pari.ramanujantau(1000))
import time
t0=time.time()
v = [int(pari.ramanujantau(n)) for n in range(1, 100)]
print("time for 100:", time.time()-t0)
print(v[:20])


tau(1)= 1
tau(2)= -24
tau(3)= 252
tau(1000)= -30328412970240000
time for 100: 0.0004246234893798828
[1, -24, 252, -1472, 4830, -6048, -16744, 84480, -113643, -115920, 534612, -370944, -577738, 401856, 1217160, 987136, -6905934, 2727432, 10661420, -7109760]


In [20]:

# Excellent! cypari2 has ramanujantau. Now compute tau for primes up to 1e6.
# Then build a_F9 multiplicatively (analytic normalized).
import time
from sympy import sieve

# Get primes up to N_MAX
t0 = time.time()
primes = list(sieve.primerange(2, N_MAX+1))
print(f"# primes up to {N_MAX}:", len(primes), " in", time.time()-t0,"s")


# primes up to 1000000: 78498 in 0.048804283142089844 s


In [21]:

# Compute tau(p) for each prime up to 1e6.
# This will likely take some time but should complete.
import time
t0 = time.time()
# Test speed on first 1000 primes
sample_primes = primes[:1000]
taus_sample = [int(pari.ramanujantau(p)) for p in sample_primes]
print(f"1000 tau computations: {time.time()-t0:.2f}s, last p={sample_primes[-1]}, tau={taus_sample[-1]}")


1000 tau computations: 0.01s, last p=7919, tau=-4467161474022023509680


In [22]:

# Very fast! Compute all 78498 primes
t0 = time.time()
tau_p_arr = np.empty(len(primes), dtype=object)
for i, p in enumerate(primes):
 tau_p_arr[i] = int(pari.ramanujantau(p))
print(f"All {len(primes)} tau values computed in {time.time()-t0:.2f}s")


TimeoutError: Code execution timed out after 900.0 seconds

In [23]:

# Timed out. cypari2 individually is too slow at scale because of overhead per call.
# Let me batch the call inside a single PARI command.
import time
t0=time.time()
# Build a PARI vector of taus
res = pari("vector(1000, n, ramanujantau(prime(n)))")
print("first batch (primes 1..1000):", time.time()-t0)
print(len(res), int(res[0]), int(res[999]))


first batch (primes 1..1000): 0.008432626724243164
1000 -24 -4467161474022023509680


In [24]:

# Batched is fast. Try the whole 78498 in one call.
import time
t0 = time.time()
res = pari(f"vector({len(primes)}, n, ramanujantau(prime(n)))")
print(f"Computed {len(primes)} taus in PARI in {time.time()-t0:.2f}s")


TimeoutError: Code execution timed out after 900.0 seconds

In [25]:

# Timed out likely by single primes with very large p. Each tau(p) for p ~ 1e6 takes longer.
# Let's measure tau(largest_prime).
import time
p_big = primes[-1]
t0=time.time()
v = pari.ramanujantau(p_big)
print("tau(", p_big, ") in", time.time()-t0,"s, value digits:", len(str(int(v))))


TimeoutError: Code execution timed out after 900.0 seconds

In [26]:

# Pari hung. Restart kernel and pivot strategy. We have very limited time.
# Key pivot: use a smaller N for modular forms. 
# Use traces vector from LMFDB (1000 entries) for F9, F10, F11. 
# Build coefficients up to N=1000 or so for F9, F10, F11.
# This is a documented limitation per the research objective: "If retrieval fails for any class, 
# proceed with the largest possible subset and document the exclusion."
# 
# Better plan: exclude F9, F10, F11 from the analysis (the modular L-functions). 
# This is documented in the prior research as a recurring issue (r15).
# Then we have 8 classes available: F1, F2, F4, F5p, F5m, F6, F7, F12.
# Of these, train on F1, F4, F6 (per protocol), test on F2, F5p, F5m, F7, F12 = 5 classes.
# 
# This is a smaller test set than ideal but allows valid execution.
print("Pivot: compute S_k using N=10^5 for speed, exclude F9-F11 due to retrieval failure.")


TimeoutError: Code execution timed out after 3.0 seconds